# 🧠 What's Next? - LSTM Next Word Predictor (Google Colab Training)
Train a Deep Learning LSTM model on quote text data, generate word sequences, and export `.h5` model and tokenizer for the web frontend.

### Step 1: Upload Dataset
Run the cell below to upload `qoute_dataset.csv` (or use the sample fallback).

In [ ]:
import os
import pandas as pd
import numpy as np
import string
import pickle
import matplotlib.pyplot as plt

# Check for uploaded dataset or upload manually
dataset_path = 'qoute_dataset.csv'
if not os.path.exists(dataset_path) and not os.path.exists('/content/qoute_dataset.csv'):
    try:
        from google.colab import files
        print("Please upload your quote_dataset.csv file:")
        uploaded = files.upload()
        dataset_path = list(uploaded.keys())[0]
    except Exception:
        print("Running outside colab upload prompt, checking standard paths...")
else:
    if os.path.exists('/content/qoute_dataset.csv'):
        dataset_path = '/content/qoute_dataset.csv'

print(f"Using dataset: {dataset_path}")

### Step 2: Load and Clean Text

In [ ]:
df = pd.read_csv(dataset_path)
print("Dataset preview:")
print(df.head())

# Identify quotes column
candidate_cols = [c for c in df.columns if c.lower() in ['quote', 'quotes', 'text', 'sentence']]
text_col = candidate_cols[0] if candidate_cols else df.columns[0]
print(f"Selected text column: '{text_col}'")

# Lowercase & remove punctuation
translator = str.maketrans('', '', string.punctuation)
quotes = df[text_col].dropna().astype(str)
quotes_cleaned = quotes.apply(lambda x: x.lower().translate(translator).strip())
quotes_cleaned = quotes_cleaned[quotes_cleaned != ""].tolist()

print(f"Total clean sentences: {len(quotes_cleaned)}")
for i in range(min(5, len(quotes_cleaned))):
    print(f" {i+1}. {quotes_cleaned[i]}")

### Step 3: Tokenization & Sequence Generation

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

max_vocab = 10000
tokenizer = Tokenizer(num_words=max_vocab, oov_token="<unk>")
tokenizer.fit_on_texts(quotes_cleaned)

vocab_size = min(len(tokenizer.word_index) + 1, max_vocab)
print(f"Unique vocabulary size (with padding): {vocab_size}")

# Convert sentences into integer sequences
sequences = tokenizer.texts_to_sequences(quotes_cleaned)

# Generate N-gram pairs for training (X = preceding words, y = next target word)
X = []
y = []
for seq in sequences:
    for i in range(1, len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

print(f"Total training pairs: {len(X)}")
max_len = max([len(x) for x in X])
print(f"Max input sequence length: {max_len}")

# Pre-pad sequences with zeros
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

# One-hot encode targets
y = np.array(y)
y_one_hot = to_categorical(y, num_classes=vocab_size)
print(f"X shape: {X_padded.shape}, y shape: {y_one_hot.shape}")

### Step 4: Build & Compile LSTM Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D

embedding_dim = 100
lstm_units = 150

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    SpatialDropout1D(0.2),
    LSTM(units=lstm_units, return_sequences=False, dropout=0.2),
    Dense(units=vocab_size, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

### Step 5: Train LSTM Model

In [ ]:
epochs = 20
batch_size = 64

history = model.fit(
    X_padded,
    y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    verbose=1
)

### Step 6: Test Inference & Generation

In [ ]:
def predict_next_word(seed_text):
    cleaned = seed_text.lower().translate(translator).strip()
    seq = tokenizer.texts_to_sequences([cleaned])[0]
    seq_padded = pad_sequences([seq], maxlen=max_len, padding='pre')
    pred = model.predict(seq_padded, verbose=0)[0]
    top_idx = np.argmax(pred)
    return tokenizer.index_word.get(top_idx, "")

def generate_text(seed_text, n_words=5):
    current = seed_text
    for _ in range(n_words):
        next_w = predict_next_word(current)
        if not next_w:
            break
        current += " " + next_w
    return current

seed = "what are you"
print(f"Seed: '{seed}' -> Next word: '{predict_next_word(seed)}'")
print(f"Generated: '{generate_text(seed, 8)}'")

### Step 7: Save & Download Artifacts for Frontend

In [ ]:
# Save model in .h5 format
model.save("lstm_model.h5")

# Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save max_len
with open("max_len.pkl", "wb") as f:
    pickle.dump(max_len, f)

print("✅ Saved lstm_model.h5, tokenizer.pkl, and max_len.pkl!")

# Auto-download in Colab
try:
    from google.colab import files
    files.download("lstm_model.h5")
    files.download("tokenizer.pkl")
    files.download("max_len.pkl")
except Exception:
    print("Files ready in current directory.")